# Efficient Frontier Demo

Sweep constraint thresholds to map out the efficient frontier — the trade-off
between objective (expected income) and constraint satisfaction.

In [1]:
import math
import time

import polars as pl
import price_contour as pc
from price_contour.frontier import frontier_summary

print(f"price_contour {pc.__version__}")

price_contour 0.1.0


## 1. Generate data

In [ ]:
def make_df(n_quotes=500, n_steps=11):
    mults = [0.80 + 0.04 * j for j in range(n_steps)]
    rows = []
    for q in range(n_quotes):
        elasticity = 1.5 + 3.5 * q / n_quotes
        base = 80.0 + 40.0 * q / n_quotes
        for j, mult in enumerate(mults):
            conversion = 1.0 / (1.0 + math.exp(elasticity * (mult - 1.0)))
            rows.append({
                "quote_id": f"Q{q:05d}",
                "scenario_index": j,
                "scenario_value": mult,
                "expected_income": base * mult * conversion,
                "volume": conversion,
                "loss_ratio": 0.6 / mult * (1.0 + 0.1 * (mult - 1.0)),
            })
    return pl.DataFrame(
        rows,
        schema={
            "quote_id": pl.Utf8,
            "scenario_index": pl.Int32,
            "scenario_value": pl.Float32,
            "expected_income": pl.Float32,
            "volume": pl.Float32,
            "loss_ratio": pl.Float32,
        },
    )


df = make_df()
print(f"Shape: {df.shape}  ({df['quote_id'].n_unique()} quotes)")

## 2. 1D Frontier: Volume constraint

Sweep the volume threshold from 85% to 100% of baseline, producing a
set of (volume_threshold, optimal_objective) trade-off points.

In [3]:
solver = pc.OnlineOptimiser(
    objective="expected_income",
    constraints={"volume": {"min": 0.90}},
    max_iter=100,
)

t0 = time.perf_counter()
result_1d = solver.frontier(
    df,
    threshold_ranges={"volume": (0.85, 1.0)},
    n_points_per_dim=8,
)
elapsed = time.perf_counter() - t0

print(f"1D frontier: {result_1d.n_points} points in {elapsed:.2f}s")
print(f"Constraint names: {result_1d.constraint_names}")
result_1d.points

ValueError: Sort failed: not found: "scenario_index" not found

### Trade-off curve

As the volume threshold tightens (higher fraction of baseline volume required),
the achievable objective decreases — revealing the cost of retention.

In [ ]:
pts = result_1d.points.sort("threshold_volume")
print(f"{'Volume threshold':>18s} {'Objective':>14s} {'Lambda':>10s} {'Iters':>6s}")
print("-" * 52)
for row in pts.iter_rows(named=True):
    print(
        f"{row['threshold_volume']:>18.2f}"
        f" {row['total_objective']:>14,.2f}"
        f" {row['lambda_volume']:>10.4f}"
        f" {row['iterations']:>6d}"
    )

## 3. 2D Frontier: Volume + Loss Ratio

Sweep both constraints simultaneously, producing an N×N grid of frontier points.

In [ ]:
solver_2d = pc.OnlineOptimiser(
    objective="expected_income",
    constraints={
        "volume": {"min": 0.90},
        "loss_ratio": {"max": 1.05},
    },
    max_iter=100,
)

t0 = time.perf_counter()
result_2d = solver_2d.frontier(
    df,
    threshold_ranges={
        "volume": (0.85, 1.0),
        "loss_ratio": (0.95, 1.10),
    },
    n_points_per_dim=5,
)
elapsed = time.perf_counter() - t0

print(f"2D frontier: {result_2d.n_points} points in {elapsed:.2f}s")
print(f"(Expected: 5 × 5 = 25 points)")
result_2d.points.head(10)

In [ ]:
print("Frontier columns:")
for col in result_2d.points.columns:
    print(f"  {col}: {result_2d.points[col].dtype}")

## 4. Frontier summary (MLflow-ready)

Select one point from the frontier and package it for logging.

In [ ]:
import json

# Select the middle point
mid_idx = result_2d.n_points // 2
summary = frontier_summary(result_2d, selected_index=mid_idx)

print("=== params ===")
for k, v in summary["params"].items():
    print(f"  {k}: {v!r}")

print("\n=== metrics ===")
for k, v in summary["metrics"].items():
    print(f"  {k}: {v:.4f}")

print(f"\n=== artifacts ===")
print(f"  frontier DataFrame: {summary['artifacts']['frontier'].shape}")

## 5. Warm-start effect

The frontier solver uses nearest-neighbour ordering + warm-start.
Compare average iterations to verify warm-start is helping.

In [ ]:
pts_df = result_1d.points
avg_iters = pts_df["iterations"].mean()
max_iters = pts_df["iterations"].max()
n_converged = pts_df.filter(pl.col("converged")).shape[0]

print(f"1D frontier stats ({result_1d.n_points} points):")
print(f"  Avg iterations:  {avg_iters:.1f}")
print(f"  Max iterations:  {max_iters}")
print(f"  Points converged: {n_converged}/{result_1d.n_points}")